## Recommended Training Configuration
Use this notebook as-is with the defaults in Cell 4.
- The model is trained only on train records.
- Validation and test records are used only for evaluation.
- Split overlap is blocked by default unless you explicitly pass --allow_overlap.

# Train Prompt Injection Classifier on Kaggle (T4x2)
This notebook installs dependencies, copies training assets from Kaggle input, trains the model, and zips artifacts for download.

In [ ]:
# 1) Install dependencies
!pip -q install --upgrade pip
!pip -q install transformers>=4.41.0 torch>=2.6.0 scikit-learn>=1.3.0 tqdm>=4.65.0 numpy>=1.24.0 sentencepiece>=0.1.99 safetensors>=0.4.3 hf_xet

In [ ]:
# 2) Locate assets from /kaggle/input and copy to /kaggle/working
import shutil
from pathlib import Path

input_root = Path('/kaggle/input')
working_root = Path('/kaggle/working')

dataset_file_name = 'cleaned_augmented_neuralchemy_dataset.jsonl'
script_file_name = 'train_classifier_kaggle.py'

dataset_candidates = list(input_root.rglob(dataset_file_name))
script_candidates = list(input_root.rglob(script_file_name))

if not dataset_candidates:
    raise FileNotFoundError(f'Could not find {dataset_file_name} under /kaggle/input')
if not script_candidates:
    raise FileNotFoundError(f'Could not find {script_file_name} under /kaggle/input')

dataset_src = dataset_candidates[0]
script_src = script_candidates[0]

dataset_dst = working_root / dataset_file_name
script_dst = working_root / script_file_name

shutil.copy2(dataset_src, dataset_dst)
shutil.copy2(script_src, script_dst)

print('Dataset source    :', dataset_src)
print('Train script src  :', script_src)
print('Dataset copy      :', dataset_dst)
print('Train script copy :', script_dst)
print('Done.')

In [ ]:
# 3) (Optional) Confirm GPU availability
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU count     :', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))

In [ ]:
# 4) Train model (single run, same kernel process)
import sys
from pathlib import Path

script_path = '/kaggle/working/train_classifier_kaggle.py'
data_path = '/kaggle/working/cleaned_augmented_neuralchemy_dataset.jsonl'
output_dir = '/kaggle/working/saved_model'
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Run trainer in the same Python process (no subprocess).
import train_classifier_kaggle as trainer

sys.argv = [
    'train_classifier_kaggle.py',
    '--data', data_path,
    '--output_dir', output_dir,
    '--model_name', 'microsoft/deberta-v3-small',
    '--epochs', '5',
    '--min_epochs', '2',
    '--patience', '2',
    '--batch', '8',
    '--grad_accum', '2',
    '--maxlen', '256',
    '--lr', '8e-6',
    '--dropout', '0.3',
    '--label_smoothing', '0.08',
    '--log_every', '25'
 ]

trainer.main()

In [ ]:
# 5) Zip artifacts for download
import os
import shutil
from pathlib import Path

output_dir = Path('/kaggle/working/saved_model')
zip_base = '/kaggle/working/saved_model_artifacts'
zip_path = zip_base + '.zip'

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(zip_base, 'zip', output_dir)

print('Created zip:', zip_path)
print('Saved model directory contents:')
for p in sorted(output_dir.rglob('*')):
    if p.is_file():
        print('-', p)

## Download outputs
After run completes, open the right-side **Output** panel in Kaggle and download:
- `/kaggle/working/saved_model_artifacts.zip`
- or the folder `/kaggle/working/saved_model/`